#  Statistical Modeling 

## Overview

This notebook performs the statistical analysis for the GenAI adoption and usage study. It estimates regression models and calculates marginal effects. 

**Analysis Sections:**

1. **Setup** - Import libraries, load data, validate variables
2. **GenAI Adoption** - Logistic regression analyzing chatbot adoption
   - Odds ratios and significance tests
   - Average Marginal Effects (AME) by age group
   - Sequential decomposition (Baseline → +Demographics → +Education → +Experience → +Tech Literacy → +STEM Education)
3. **Intent Usage** - OLS regression for usage frequency across 6 intent types
   - Information Retrieval, Problem Solving, Learning, Content Creation, Leisure, Creativity
   - AME by intent with bootstrap confidence intervals (1000 iterations)
4. **Gender Decomposition** - Sequential models examining gender gap evolution
   - Pooled usage model with intent fixed effects
   - Intent-specific decomposition (6 panels showing progression across control specifications)

**Key Outputs:**

All results saved to `../results/` directory:
- **TSV files:** Model summaries, AME estimates, decomposition results
- **LaTeX tables:** Odds ratios, OLS regression, sequential decomposition (adoption, pooled usage, by-intent)

## 1. Setup

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.formula.api as smf
from scipy.stats import norm
from stargazer.stargazer import Stargazer

In [ ]:
# Set up paths
notebook_dir = Path(".").resolve()
project_root = notebook_dir.parent
data_path = project_root / "survey_clean_var.tsv"
results_dir = project_root / "results"

In [ ]:
# Load data with pre-engineered variables
df_clean = pd.read_csv(data_path, sep="\t", encoding='utf-8')

print(f"✓ Dataset loaded: {len(df_clean)} respondents")
print(f"✓ Columns: {len(df_clean.columns)}")

In [ ]:
# Validate required variables are present
required_vars = [
    'chatbot_user', 'LT_exp', 'lt_lit', 
    'EducationGroup', 'Education_STEM', 'GeographyGroup', 
    'GenderGroup', 'IncomeGroup', 'AgeGroup',
    'InfoRetrieval_freq', 'ProblemSolving_freq', 'Learning_freq',
    'ContentCreation_freq', 'Entertainment_freq', 'Creativity_freq'
]

missing_vars = [var for var in required_vars if var not in df_clean.columns]

if missing_vars:
    print(f"⚠️  Missing variables: {missing_vars}")
    raise ValueError(f"Required variables missing: {missing_vars}")
else:
    print(f"✓ All required engineered variables present")

In [ ]:
# Prepare analysis dataset
# Filter out "Neither" gender and create binary chatbot user variable
df_analysis = df_clean[df_clean['GenderGroup'].isin(['Man', 'Woman'])].copy()
df_analysis['chatbot_user_binary'] = (df_analysis['chatbot_user'] == 'Yes').astype(int)

print(f"✓ Analysis dataset prepared: {len(df_analysis)} observations")

In [ ]:
# Set reference categories for categorical variables
df_analysis['GenderGroup'] = pd.Categorical(
    df_analysis['GenderGroup'],
    categories=['Man', 'Woman'],  # Man is reference
    ordered=False
)

df_analysis['AgeGroup'] = pd.Categorical(
    df_analysis['AgeGroup'],
    categories=['18-34', '35-54', '55-64', '65+'],  # 18-34 is reference
    ordered=False
)

df_analysis['GeographyGroup'] = pd.Categorical(
    df_analysis['GeographyGroup'],
    categories=['North', 'Centre', 'South and Islands', 'Abroad'],  # North is reference
    ordered=False
)

df_analysis['IncomeGroup'] = pd.Categorical(
    df_analysis['IncomeGroup'],
    categories=['Lower', 'Mid', 'Higher'],  # Lower is reference
    ordered=False
)

df_analysis['EducationGroup'] = pd.Categorical(
    df_analysis['EducationGroup'],
    categories=['Non-graduates', 'Graduates'],  # Non-graduates is reference
    ordered=False
)

## 2. GenAI Adoption - Logistic Regression

In [ ]:
# Fit logistic regression model for chatbot adoption
formula_adoption = 'chatbot_user_binary ~ C(GenderGroup) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + LT_exp + C(EducationGroup) + lt_lit'
model_adoption = smf.logit(formula_adoption, data=df_analysis).fit(disp=0)

# print("=" * 80)
# print("LOGISTIC REGRESSION MODEL - CHATBOT ADOPTION")
# print("=" * 80)
# print(model_adoption.summary())
# print("\n")

# Save model summary
model_summary = model_adoption.summary2().tables[1]
model_summary.to_csv(results_dir / 'logistic_adoption_summary.tsv', sep='\t')
print("Model summary saved to: results/logistic_adoption_summary.tsv")

### Odds Ratio

In [ ]:
# Calculate odds ratios with confidence intervals
coef_table = model_adoption.summary2().tables[1]

# Calculate odds ratios and confidence intervals
odds_ratios = np.exp(coef_table['Coef.'])
ci_lower = np.exp(coef_table['[0.025'])
ci_upper = np.exp(coef_table['0.975]'])
se_or = odds_ratios * coef_table['Std.Err.']

# Create structured dataframe (excluding intercept)
logit_r_adoption = pd.DataFrame({
    'variable': [
        'Woman', 
        '35-54', '55-64', '65+',
        'Centre', 'South&Islands', 'Abroad',
        'Mid', 'Higher',
        'Graduates',
        'LT experience', 
        'LT literacy'
    ],
    'category': [
        'Gender (Man)',
        'Age (18-34)', 'Age (18-34)', 'Age (18-34)',
        'Geography (North)', 'Geography (North)', 'Geography (North)',
        'Income (Lower)', 'Income (Lower)',
        'Education (Non-graduates)',
        'LT experience', 
        'LT literacy'
    ],
    'odds_ratio': [
        odds_ratios['C(GenderGroup)[T.Woman]'],
        odds_ratios['C(AgeGroup)[T.35-54]'],
        odds_ratios['C(AgeGroup)[T.55-64]'],
        odds_ratios['C(AgeGroup)[T.65+]'],
        odds_ratios['C(GeographyGroup)[T.Centre]'],
        odds_ratios['C(GeographyGroup)[T.South and Islands]'],
        odds_ratios['C(GeographyGroup)[T.Abroad]'],
        odds_ratios['C(IncomeGroup)[T.Mid]'],
        odds_ratios['C(IncomeGroup)[T.Higher]'],
        odds_ratios['C(EducationGroup)[T.Graduates]'],
        odds_ratios['LT_exp'],
        odds_ratios['lt_lit']
    ],
    'standard_error': [
        se_or['C(GenderGroup)[T.Woman]'],
        se_or['C(AgeGroup)[T.35-54]'],
        se_or['C(AgeGroup)[T.55-64]'],
        se_or['C(AgeGroup)[T.65+]'],
        se_or['C(GeographyGroup)[T.Centre]'],
        se_or['C(GeographyGroup)[T.South and Islands]'],
        se_or['C(GeographyGroup)[T.Abroad]'],
        se_or['C(IncomeGroup)[T.Mid]'],
        se_or['C(IncomeGroup)[T.Higher]'],
        se_or['C(EducationGroup)[T.Graduates]'],
        se_or['LT_exp'],
        se_or['lt_lit']
    ],
    'ci_lower': [
        ci_lower['C(GenderGroup)[T.Woman]'],
        ci_lower['C(AgeGroup)[T.35-54]'],
        ci_lower['C(AgeGroup)[T.55-64]'],
        ci_lower['C(AgeGroup)[T.65+]'],
        ci_lower['C(GeographyGroup)[T.Centre]'],
        ci_lower['C(GeographyGroup)[T.South and Islands]'],
        ci_lower['C(GeographyGroup)[T.Abroad]'],
        ci_lower['C(IncomeGroup)[T.Mid]'],
        ci_lower['C(IncomeGroup)[T.Higher]'],
        ci_lower['C(EducationGroup)[T.Graduates]'],
        ci_lower['LT_exp'],
        ci_lower['lt_lit']
    ],
    'ci_upper': [
        ci_upper['C(GenderGroup)[T.Woman]'],
        ci_upper['C(AgeGroup)[T.35-54]'],
        ci_upper['C(AgeGroup)[T.55-64]'],
        ci_upper['C(AgeGroup)[T.65+]'],
        ci_upper['C(GeographyGroup)[T.Centre]'],
        ci_upper['C(GeographyGroup)[T.South and Islands]'],
        ci_upper['C(GeographyGroup)[T.Abroad]'],
        ci_upper['C(IncomeGroup)[T.Mid]'],
        ci_upper['C(IncomeGroup)[T.Higher]'],
        ci_upper['C(EducationGroup)[T.Graduates]'],
        ci_upper['LT_exp'],
        ci_upper['lt_lit']
    ],
    'p_value': [
        coef_table.loc['C(GenderGroup)[T.Woman]', 'P>|z|'],
        coef_table.loc['C(AgeGroup)[T.35-54]', 'P>|z|'],
        coef_table.loc['C(AgeGroup)[T.55-64]', 'P>|z|'],
        coef_table.loc['C(AgeGroup)[T.65+]', 'P>|z|'],
        coef_table.loc['C(GeographyGroup)[T.Centre]', 'P>|z|'],
        coef_table.loc['C(GeographyGroup)[T.South and Islands]', 'P>|z|'],
        coef_table.loc['C(GeographyGroup)[T.Abroad]', 'P>|z|'],
        coef_table.loc['C(IncomeGroup)[T.Mid]', 'P>|z|'],
        coef_table.loc['C(IncomeGroup)[T.Higher]', 'P>|z|'],
        coef_table.loc['C(EducationGroup)[T.Graduates]', 'P>|z|'],
        coef_table.loc['LT_exp', 'P>|z|'],
        coef_table.loc['lt_lit', 'P>|z|']
    ]
})

# Calculate significance based on CI not crossing 1.0
logit_r_adoption['significant'] = ~((logit_r_adoption['ci_lower'] <= 1.0) & (logit_r_adoption['ci_upper'] >= 1.0))

print("\n" + "=" * 80)
print("ODDS RATIOS")
print("=" * 80)
print(logit_r_adoption[['variable', 'odds_ratio', 'ci_lower', 'ci_upper', 'p_value', 'significant']])

# Save odds ratios
logit_r_adoption.to_csv(results_dir / 'logit_r_adoption.tsv', sep='\t', index=False)
print("\nOdds ratios saved to: results/logit_r_adoption.tsv")

### LaTeX Table - Odds Ratios

In [ ]:
# Generate and save LaTeX table
cols = ['variable', 'odds_ratio', 'ci_lower', 'ci_upper', 'p_value', 'significant']
latex_table = logit_r_adoption[cols].to_latex(
    index=False,
    float_format="%.3f",
    caption='Logistic Regression: Odds Ratios for Chatbot Adoption',
    label='tab:odds_ratios',
    escape=False
)

# print("\n" + "=" * 80)
# print("LATEX TABLE - ODDS RATIOS")
# print("=" * 80)
# print(latex_table)

with open(results_dir / 'table_odds_ratios.tex', 'w') as f:
    f.write(latex_table)
print("LaTeX table saved to: results/table_odds_ratios.tex")

### Gender AME by Age


In [ ]:
# Calculate Average Marginal Effects (AME) by age group
print("\n" + "=" * 80)
print("AVERAGE MARGINAL EFFECTS - ADOPTION BY AGE GROUP")
print("=" * 80)

age_groups = ['18-34', '35-54', '55-64', '65+']
gender_gaps_ame_adoption = []
ci_lower_ame_adoption = []
ci_upper_ame_adoption = []
n_bootstrap = 1000
np.random.seed(42)

for age in age_groups:
    print(f"\nProcessing age group: {age}")
    age_subset = df_analysis[df_analysis['AgeGroup'] == age].copy()
    print(f"  N = {len(age_subset)}")
    
    # Create counterfactual datasets
    data_woman = age_subset.copy()
    data_woman['GenderGroup'] = 'Woman'
    data_man = age_subset.copy()
    data_man['GenderGroup'] = 'Man'
    
    # Predict probabilities for both scenarios
    pred_woman = model_adoption.predict(data_woman)
    pred_man = model_adoption.predict(data_man)
    
    # Calculate Average Marginal Effect
    ame = (pred_woman - pred_man).mean()
    gender_gaps_ame_adoption.append(ame)
    print(f"  AME (Woman - Man): {ame:.4f}")
    
    # Bootstrap for confidence intervals
    bootstrap_gaps = []
    for b in range(n_bootstrap):
        sample_idx = np.random.choice(len(df_analysis), len(df_analysis), replace=True)
        sample_data = df_analysis.iloc[sample_idx]
        try:
            boot_model = smf.logit(formula_adoption, data=sample_data).fit(disp=0, maxiter=50, warn_convergence=False)
            boot_pred_woman = boot_model.predict(data_woman)
            boot_pred_man = boot_model.predict(data_man)
            boot_ame = (boot_pred_woman - boot_pred_man).mean()
            bootstrap_gaps.append(boot_ame)
        except:
            continue
    
    # Calculate 95% CI
    if len(bootstrap_gaps) > 100:
        ci_lower_ame_adoption.append(np.percentile(bootstrap_gaps, 2.5))
        ci_upper_ame_adoption.append(np.percentile(bootstrap_gaps, 97.5))
        print(f"  95% CI: [{ci_lower_ame_adoption[-1]:.4f}, {ci_upper_ame_adoption[-1]:.4f}]")
        print(f"  Successful bootstrap iterations: {len(bootstrap_gaps)}/{n_bootstrap}")
    else:
        ci_lower_ame_adoption.append(np.nan)
        ci_upper_ame_adoption.append(np.nan)
        print(f"  WARNING: Only {len(bootstrap_gaps)} successful iterations - CI set to NaN")

# Save AME results
ame_adoption_df = pd.DataFrame({
    'age_group': age_groups,
    'ame': gender_gaps_ame_adoption,
    'ci_lower': ci_lower_ame_adoption,
    'ci_upper': ci_upper_ame_adoption
})
ame_adoption_df.to_csv(results_dir / 'logistic_adoption_ame_by_age.tsv', sep='\t', index=False)

print("\n" + "=" * 80)
print("SUMMARY - ADOPTION AME BY AGE")
print("=" * 80)
print(ame_adoption_df.to_string(index=False))
print("\n✓ AME results saved to: results/logistic_adoption_ame_by_age.tsv")

## 3. Intent Usage - OLS Regressions

In [ ]:
# Define intent types
intents = {
    'InfoRetrieval_freq': 'Information Retrieval',
    'ProblemSolving_freq': 'Problem Solving',
    'Learning_freq': 'Learning',
    'ContentCreation_freq': 'Content Creation',
    'Entertainment_freq': 'Leisure',
    'Creativity_freq': 'Creativity'
}

In [ ]:
# OLS Models for Usage Intensity with AME
print("=" * 80)
print("OLS REGRESSION MODELS - USAGE INTENSITY BY INTENT")
print("=" * 80)

model2_results_ame_usage = []
usage_models = {}
n_bootstrap = 1000
np.random.seed(42)

for intent_col, intent_label in intents.items():
    if intent_col not in df_analysis.columns:
        print(f"\n⚠️  Skipping {intent_label}: column '{intent_col}' not found")
        continue
    
    print(f"\n{'='*80}")
    print(f"MODEL: {intent_label.upper()}")
    print(f"{'='*80}")
    
    # Filter to non-missing values for this intent
    df_intent = df_analysis[df_analysis[intent_col].notna()].copy()
    print(f"N = {len(df_intent)}\n")
    
    # Fit OLS model
    formula_usage = f'{intent_col} ~ C(GenderGroup) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + LT_exp + C(EducationGroup) + lt_lit'
    
    try:
        model_usage = smf.ols(formula_usage, data=df_intent).fit()
        usage_models[intent_label] = model_usage
        
        # Print model summary
        print(model_usage.summary())
        print("\n")
        
        # Save individual model summary
        model_summary = model_usage.summary2().tables[1]
        filename = f'ols_{intent_label.lower().replace(" ", "_")}_summary.tsv'
        model_summary.to_csv(results_dir / filename, sep='\t')
        print(f"✓ Model summary saved to: results/{filename}")
        
        # Calculate AME
        data_woman = df_intent.copy()
        data_woman['GenderGroup'] = 'Woman'
        data_man = df_intent.copy()
        data_man['GenderGroup'] = 'Man'
        
        pred_woman = model_usage.predict(data_woman)
        pred_man = model_usage.predict(data_man)
        ame = (pred_woman - pred_man).mean()
        
        print(f"AME (Woman - Man): {ame:.4f}")
        
        # Bootstrap for CI
        bootstrap_ames = []
        for b in range(n_bootstrap):
            sample_idx = np.random.choice(len(df_intent), len(df_intent), replace=True)
            sample_data = df_intent.iloc[sample_idx]
            try:
                boot_model = smf.ols(formula_usage, data=sample_data).fit()
                boot_pred_woman = boot_model.predict(data_woman)
                boot_pred_man = boot_model.predict(data_man)
                boot_ame = (boot_pred_woman - boot_pred_man).mean()
                bootstrap_ames.append(boot_ame)
            except:
                continue
        
        # Calculate CI and p-value
        if len(bootstrap_ames) > 100:
            ci_lower_intent = np.percentile(bootstrap_ames, 2.5)
            ci_upper_intent = np.percentile(bootstrap_ames, 97.5)
            
            # Calculate p-value from bootstrap distribution
            if ame > 0:
                pval = 2 * np.mean(np.array(bootstrap_ames) <= 0)
            else:
                pval = 2 * np.mean(np.array(bootstrap_ames) >= 0)
            pval = min(pval, 1.0)
            
            print(f"95% CI: [{ci_lower_intent:.4f}, {ci_upper_intent:.4f}]")
            print(f"p-value: {pval:.4f}\n")
        else:
            ci_lower_intent = np.nan
            ci_upper_intent = np.nan
            pval = np.nan
            print(f"WARNING: Only {len(bootstrap_ames)} successful iterations - CI set to NaN\n")
        
        model2_results_ame_usage.append({
            'intent': intent_label,
            'gap': ame,
            'ci_lower': ci_lower_intent,
            'ci_upper': ci_upper_intent,
            'pval': pval
        })
        
    except Exception as e:
        print(f"⚠️  Model failed: {e}\n")

# Sort by gap size
model2_results_ame_usage.sort(key=lambda x: x['gap'])

# Save AME results
ame_usage_df = pd.DataFrame(model2_results_ame_usage)
ame_usage_df.to_csv(results_dir / 'ols_usage_intensity_ame_by_intent.tsv', sep='\t', index=False)

print("\n" + "=" * 80)
print("SUMMARY - USAGE INTENSITY AME BY INTENT")
print("=" * 80)
for result in model2_results_ame_usage:
    sig = "***" if result['pval'] < 0.001 else "**" if result['pval'] < 0.01 else "*" if result['pval'] < 0.05 else ""
    if not np.isnan(result['ci_lower']):
        print(f"  {result['intent']:<25} {result['gap']:>7.4f} [{result['ci_lower']:>7.4f}, {result['ci_upper']:>7.4f}] {sig}")
    else:
        print(f"  {result['intent']:<25} {result['gap']:>7.4f} [CI unavailable]")

print("\n✓ All OLS model summaries saved to: results/ols_*_summary.tsv")
print("✓ AME results saved to: results/ols_usage_intensity_ame_by_intent.tsv")

### LaTeX Tables - OLS Results

In [ ]:
# LaTeX Table: OLS Regression Results using Stargazer
stargazer = Stargazer(list(usage_models.values()))
stargazer.title('OLS Regression Models - Usage Frequency by Intent')
stargazer.custom_columns(list(usage_models.keys()), [1]*len(usage_models))

# # Generate LaTeX
# latex_output = stargazer.render_latex()
# print("\n" + "=" * 80)
# print("LATEX TABLE - OLS REGRESSION RESULTS")
# print("=" * 80)
# print(latex_output)

# Save to file
with open(results_dir / 'table_ols_regression.tex', 'w') as f:
    f.write(latex_output)
print("\n✓ LaTeX table saved to: results/table_ols_regression.tex")

## 4. Gender Gap Decomposition - Adoption

In [ ]:
# Sequential decomposition showing what explains the gender gap in adoption
print("=" * 80)
print("SEQUENTIAL DECOMPOSITION - GENDER GAP IN CHATBOT ADOPTION")
print("=" * 80)

# Define sequential models
models_spec_adoption = {
    'Model 1: Baseline': 
        'chatbot_user_binary ~ C(GenderGroup)',
    
    'Model 2: + Demographics': 
        'chatbot_user_binary ~ C(GenderGroup) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup)',
    
    'Model 3: + Education': 
        'chatbot_user_binary ~ C(GenderGroup) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + C(EducationGroup)',
    
    'Model 4: + Experience': 
        'chatbot_user_binary ~ C(GenderGroup) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + C(EducationGroup) + LT_exp',
    
    'Model 5: + Tech Literacy': 
        'chatbot_user_binary ~ C(GenderGroup) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + C(EducationGroup) + LT_exp + lt_lit',
    
    'Model 6: + STEM Education': 
        'chatbot_user_binary ~ C(GenderGroup) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + C(EducationGroup) + LT_exp + lt_lit + C(Education_STEM)'
}

results_decomp_adoption = []

for model_name, formula in models_spec_adoption.items():
    print(f"\n{model_name}")
    print("-" * 80)
    
    model = smf.logit(formula, data=df_analysis).fit(disp=0)
    
    # Get the coefficient for Woman
    gender_coef_name = [name for name in model.params.index if 'GenderGroup' in name and 'Woman' in name]
    
    if gender_coef_name:
        coef = model.params[gender_coef_name[0]]
        se = model.bse[gender_coef_name[0]]
        pval = model.pvalues[gender_coef_name[0]]
        
        # Calculate marginal effect
        pred_probs = model.predict(df_analysis)
        marginal_effect = coef * (pred_probs * (1 - pred_probs)).mean()
        
        results_decomp_adoption.append({
            'Model': model_name,
            'Coefficient': coef,
            'Std Error': se,
            'P-value': pval,
            'Marginal Effect': marginal_effect,
            'Pseudo R²': model.prsquared,
            'N': int(model.nobs)
        })
        
        stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f"  Woman Coefficient: {coef:.4f}{stars} (SE: {se:.4f})")
        print(f"  Marginal Effect:   {marginal_effect:.4f}")
        print(f"  Pseudo R²:         {model.prsquared:.4f}")

# Save decomposition results
df_decomp_adoption = pd.DataFrame(results_decomp_adoption)
df_decomp_adoption.to_csv(results_dir / 'decomposition_adoption.tsv', sep='\t', index=False)

print("\n\n" + "=" * 80)
print("SUMMARY - SEQUENTIAL DECOMPOSITION (ADOPTION)")
print("=" * 80)
print(f"\n{'Model':<40} {'Woman Coef':<15} {'Marginal Eff':<15} {'Pseudo R²':<12}")
print("-" * 80)

for _, row in df_decomp_adoption.iterrows():
    stars = '***' if row['P-value'] < 0.01 else '**' if row['P-value'] < 0.05 else '*' if row['P-value'] < 0.1 else ''
    model_short = row['Model'].replace('Model ', 'M')
    print(f"{model_short:<40} {row['Coefficient']:>7.4f}{stars:<6} {row['Marginal Effect']:>10.4f}    {row['Pseudo R²']:>8.4f}")

print("\n*** p<0.01, ** p<0.05, * p<0.1")
print("\n✓ Decomposition results saved to: results/decomposition_adoption.tsv")

### LaTeX Table - Adoption Decomposition

In [ ]:
# Generate LaTeX table for Adoption Decomposition
def format_coef_stars(coef, pval, decimals=3):
    """Format coefficient with significance stars"""
    stars = '^{***}' if pval < 0.01 else '^{**}' if pval < 0.05 else '^{*}' if pval < 0.1 else ''
    return f"{coef:.{decimals}f}{stars}"

def format_se(se, decimals=3):
    """Format standard error in parentheses"""
    return f"({se:.{decimals}f})"

latex_decomp_adopt = []
latex_decomp_adopt.append("\\begin{table}[htbp]")
latex_decomp_adopt.append("\\centering")
latex_decomp_adopt.append("\\caption{Sequential Decomposition of Gender Gap in Chatbot Adoption}")
latex_decomp_adopt.append("\\label{tab:decomp_adoption}")
latex_decomp_adopt.append("\\begin{tabular}{lcccccc}")
latex_decomp_adopt.append("\\toprule")
latex_decomp_adopt.append("& (1) & (2) & (3) & (4) & (5) & (6) \\\\")
latex_decomp_adopt.append("& Baseline & +Demo & +Edu & +Exp & +TechLit & +STEM \\\\")
latex_decomp_adopt.append("\\midrule")

# Female coefficient row
female_row = "Woman & "
for _, row in df_decomp_adoption.iterrows():
    female_row += format_coef_stars(row['Coefficient'], row['P-value']) + " & "
female_row = female_row.rstrip(" & ") + " \\\\"
latex_decomp_adopt.append(female_row)

# SE row
se_row = " & "
for _, row in df_decomp_adoption.iterrows():
    se_row += format_se(row['Std Error']) + " & "
se_row = se_row.rstrip(" & ") + " \\\\"
latex_decomp_adopt.append(se_row)

latex_decomp_adopt.append("\\midrule")

# Marginal Effect row
me_row = "Marginal Effect & "
for _, row in df_decomp_adoption.iterrows():
    me_row += f"{row['Marginal Effect']:.3f} & "
me_row = me_row.rstrip(" & ") + " \\\\"
latex_decomp_adopt.append(me_row)

# Pseudo R² row
r2_row = "Pseudo $R^2$ & "
for _, row in df_decomp_adoption.iterrows():
    r2_row += f"{row['Pseudo R²']:.3f} & "
r2_row = r2_row.rstrip(" & ") + " \\\\"
latex_decomp_adopt.append(r2_row)

# N row
n_row = "N & "
for _, row in df_decomp_adoption.iterrows():
    n_row += f"{row['N']:,} & "
n_row = n_row.rstrip(" & ") + " \\\\"
latex_decomp_adopt.append(n_row)

latex_decomp_adopt.append("\\bottomrule")
latex_decomp_adopt.append("\\end{tabular}")
latex_decomp_adopt.append("\\begin{tablenotes}")
latex_decomp_adopt.append("\\small")
latex_decomp_adopt.append("\\item \\textit{Notes:} Logistic regression coefficients with standard errors in parentheses.")
latex_decomp_adopt.append("\\item Reference category: Man. Controls progressively added across columns.")
latex_decomp_adopt.append("\\item Demo = Demographics (Age, Geography, Income); Edu = Education level;")
latex_decomp_adopt.append("\\item Exp = LT Experience; TechLit = Tech Literacy; STEM = STEM Education.")
latex_decomp_adopt.append("\\item $^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.1$")
latex_decomp_adopt.append("\\end{tablenotes}")
latex_decomp_adopt.append("\\end{table}")

latex_decomp_adopt_str = "\\n".join(latex_decomp_adopt)

# print("\n" + "=" * 80)
# print("LATEX TABLE - ADOPTION DECOMPOSITION")
# print("=" * 80)
# print(latex_decomp_adopt_str)

# Save LaTeX table
with open(results_dir / 'table_decomp_adoption.tex', 'w') as f:
    f.write(latex_decomp_adopt_str)
print("\n✓ LaTeX table saved to: results/table_decomp_adoption.tex")

## 5. Gender Gap Decomposition - Usage Intensity (Pooled)

In [ ]:
# Create pooled dataset combining all intent types
print("=" * 80)
print("SEQUENTIAL DECOMPOSITION - USAGE INTENSITY (POOLED)")
print("=" * 80)

pooled_data = []
for intent_col, intent_label in intents.items():
    df_intent = df_analysis[df_analysis[intent_col].notna()].copy()
    df_intent['usage_freq'] = df_intent[intent_col]
    df_intent['intent_type'] = intent_label
    pooled_data.append(df_intent[['usage_freq', 'intent_type', 'GenderGroup', 'AgeGroup', 
                                    'GeographyGroup', 'IncomeGroup', 'EducationGroup', 
                                    'LT_exp', 'lt_lit', 'Education_STEM']])

df_pooled = pd.concat(pooled_data, ignore_index=True)
print(f"\nPooled dataset: N = {len(df_pooled)} observations from {df_pooled['intent_type'].nunique()} intent types\n")

# Define sequential models for pooled data
pooled_models = {
    'Model 1: Baseline': 
        'usage_freq ~ C(GenderGroup) + C(intent_type)',
    
    'Model 2: + Demographics': 
        'usage_freq ~ C(GenderGroup) + C(intent_type) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup)',
    
    'Model 3: + Education': 
        'usage_freq ~ C(GenderGroup) + C(intent_type) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + C(EducationGroup)',
    
    'Model 4: + Experience': 
        'usage_freq ~ C(GenderGroup) + C(intent_type) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + C(EducationGroup) + LT_exp',
    
    'Model 5: + Tech Literacy': 
        'usage_freq ~ C(GenderGroup) + C(intent_type) + C(AgeGroup) + C(GeographyGroup) + C(IncomeGroup) + C(EducationGroup) + LT_exp + lt_lit'
}

pooled_results = []

for model_name, formula in pooled_models.items():
    print(f"\n{model_name}")
    print("-" * 80)
    
    model = smf.ols(formula, data=df_pooled).fit()
    
    # Get coefficient for Woman
    gender_coef_name = [name for name in model.params.index if 'GenderGroup' in name and 'Woman' in name]
    
    if gender_coef_name:
        coef = model.params[gender_coef_name[0]]
        se = model.bse[gender_coef_name[0]]
        pval = model.pvalues[gender_coef_name[0]]
        
        pooled_results.append({
            'Model': model_name,
            'Coefficient': coef,
            'Std Error': se,
            'P-value': pval,
            'R²': model.rsquared,
            'Adj R²': model.rsquared_adj,
            'N': int(model.nobs)
        })
        
        stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f"  Woman Coefficient: {coef:.4f}{stars} (SE: {se:.4f})")
        print(f"  R²:                {model.rsquared:.4f}")

# Save pooled decomposition results
df_pooled_results = pd.DataFrame(pooled_results)
df_pooled_results.to_csv(results_dir / 'decomposition_usage_pooled.tsv', sep='\t', index=False)

print("\n\n" + "=" * 80)
print("SUMMARY - POOLED DECOMPOSITION (USAGE)")
print("=" * 80)
print(f"\n{'Model':<40} {'Woman Coef':<15} {'R²':<10} {'Adj R²':<10}")
print("-" * 80)

for _, row in df_pooled_results.iterrows():
    stars = '***' if row['P-value'] < 0.01 else '**' if row['P-value'] < 0.05 else '*' if row['P-value'] < 0.1 else ''
    model_short = row['Model'].replace('Model ', 'M')
    print(f"{model_short:<40} {row['Coefficient']:>7.4f}{stars:<6} {row['R²']:>6.4f}    {row['Adj R²']:>6.4f}")

print("\n*** p<0.01, ** p<0.05, * p<0.1")
print("\n✓ Pooled decomposition results saved to: results/decomposition_usage_pooled.tsv")

### LaTeX Table - Usage Decomposition (Pooled)

In [ ]:
# Generate LaTeX table for Pooled Usage Decomposition
latex_decomp_pooled = []
latex_decomp_pooled.append("\\begin{table}[htbp]")
latex_decomp_pooled.append("\\centering")
latex_decomp_pooled.append("\\caption{Sequential Decomposition of Gender Gap in Usage Intensity: Pooled Model}")
latex_decomp_pooled.append("\\label{tab:decomp_usage_pooled}")
latex_decomp_pooled.append("\\begin{tabular}{lccccc}")
latex_decomp_pooled.append("\\toprule")
latex_decomp_pooled.append("& (1) & (2) & (3) & (4) & (5) \\\\")
latex_decomp_pooled.append("& Baseline & +Demo & +Edu & +Exp & +TechLit \\\\")
latex_decomp_pooled.append("\\midrule")

# Female coefficient row
female_row = "Woman & "
for _, row in df_pooled_results.iterrows():
    female_row += format_coef_stars(row['Coefficient'], row['P-value']) + " & "
female_row = female_row.rstrip(" & ") + " \\\\"
latex_decomp_pooled.append(female_row)

# SE row
se_row = " & "
for _, row in df_pooled_results.iterrows():
    se_row += format_se(row['Std Error']) + " & "
se_row = se_row.rstrip(" & ") + " \\\\"
latex_decomp_pooled.append(se_row)

latex_decomp_pooled.append("\\midrule")

# R² row
r2_row = "$R^2$ & "
for _, row in df_pooled_results.iterrows():
    r2_row += f"{row['R²']:.3f} & "
r2_row = r2_row.rstrip(" & ") + " \\\\"
latex_decomp_pooled.append(r2_row)

# Adjusted R² row
adj_r2_row = "Adj. $R^2$ & "
for _, row in df_pooled_results.iterrows():
    adj_r2_row += f"{row['Adj R²']:.3f} & "
adj_r2_row = adj_r2_row.rstrip(" & ") + " \\\\"
latex_decomp_pooled.append(adj_r2_row)

# N row
n_row = "N & "
for _, row in df_pooled_results.iterrows():
    n_row += f"{row['N']:,} & "
n_row = n_row.rstrip(" & ") + " \\\\"
latex_decomp_pooled.append(n_row)

latex_decomp_pooled.append("\\bottomrule")
latex_decomp_pooled.append("\\end{tabular}")
latex_decomp_pooled.append("\\begin{tablenotes}")
latex_decomp_pooled.append("\\small")
latex_decomp_pooled.append("\\item \\textit{Notes:} OLS regression coefficients with standard errors in parentheses.")
latex_decomp_pooled.append("\\item Dependent variable: Usage frequency (pooled across all intent types).")
latex_decomp_pooled.append("\\item Reference category: Man. All models control for intent type fixed effects.")
latex_decomp_pooled.append("\\item Demo = Demographics (Age, Geography, Income); Edu = Education level;")
latex_decomp_pooled.append("\\item Exp = LT Experience; TechLit = Tech Literacy.")
latex_decomp_pooled.append("\\item $^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.1$")
latex_decomp_pooled.append("\\end{tablenotes}")
latex_decomp_pooled.append("\\end{table}")

latex_decomp_pooled_str = "\\n".join(latex_decomp_pooled)

# print("\n" + "=" * 80)
# print("LATEX TABLE - POOLED USAGE DECOMPOSITION")
# print("=" * 80)
# print(latex_decomp_pooled_str)

# Save LaTeX table
with open(results_dir / 'table_decomp_usage_pooled.tex', 'w') as f:
    f.write(latex_decomp_pooled_str)
print("\n✓ LaTeX table saved to: results/table_decomp_usage_pooled.tex")

# Gender Decomposition per Intent

In [ ]:
# LaTeX Table: Sequential Decomposition by Intent (6 Panels)

# Define intent types in order
intent_types = [
    'InfoRetrieval_freq',
    'ProblemSolving_freq', 
    'Learning_freq',
    'ContentCreation_freq',
    'Entertainment_freq',
    'Creativity_freq'
]

intent_labels = {
    'InfoRetrieval_freq': 'Information Retrieval',
    'ProblemSolving_freq': 'Problem Solving',
    'Learning_freq': 'Learning',
    'ContentCreation_freq': 'Content Creation',
    'Entertainment_freq': 'Leisure',
    'Creativity_freq': 'Creativity'
}

# Define sequential model specifications
seq_models = [
    {'name': 'Baseline', 'controls': ['C(GenderGroup)']},
    {'name': '+Demo', 'controls': ['C(GenderGroup)', 'C(AgeGroup)', 'C(GeographyGroup)', 'C(IncomeGroup)']},
    {'name': '+Edu', 'controls': ['C(GenderGroup)', 'C(AgeGroup)', 'C(GeographyGroup)', 'C(IncomeGroup)', 'C(EducationGroup)']},
    {'name': '+Exp', 'controls': ['C(GenderGroup)', 'C(AgeGroup)', 'C(GeographyGroup)', 'C(IncomeGroup)', 'C(EducationGroup)', 'LT_exp']},
    {'name': '+TechLit', 'controls': ['C(GenderGroup)', 'C(AgeGroup)', 'C(GeographyGroup)', 'C(IncomeGroup)', 'C(EducationGroup)', 'LT_exp', 'lt_lit']},
    {'name': '+TechEdu', 'controls': ['C(GenderGroup)', 'C(AgeGroup)', 'C(GeographyGroup)', 'C(IncomeGroup)', 'C(EducationGroup)', 'LT_exp', 'lt_lit', 'C(Education_STEM)']}
]

print("\n" + "=" * 80)
print("SEQUENTIAL DECOMPOSITION BY INTENT - Fitting Models")
print("=" * 80)

# Store all results
decomp_results = {}

for intent_var in intent_types:
    intent_label = intent_labels[intent_var]
    print(f"\n--- {intent_label} ({intent_var}) ---")
    
    decomp_results[intent_var] = []
    
    for spec in seq_models:
        # Build formula WITH intercept and C() wrappers already in place
        formula = f"{intent_var} ~ {' + '.join(spec['controls'])}"
        
        # Fit model
        model = smf.ols(formula, data=df_analysis).fit()
        
        # Extract Woman coefficient
        gender_coef_name = [n for n in model.params.index if 'GenderGroup' in n and 'Woman' in n]
        if gender_coef_name:
            coef = model.params[gender_coef_name[0]]
            se_coef = model.bse[gender_coef_name[0]]
            pval_coef = model.pvalues[gender_coef_name[0]]
        else:
            coef = np.nan
            se_coef = np.nan
            pval_coef = np.nan
        
        # For OLS, AME = coefficient, SE(AME) = SE(coefficient)
        ame = coef
        se_ame = se_coef
        
        # Store results
        decomp_results[intent_var].append({
            'spec': spec['name'],
            'coef': coef,
            'se_coef': se_coef,
            'pval_coef': pval_coef,
            'ame': ame,
            'se_ame': se_ame,
            'r2': model.rsquared,
            'n': int(model.nobs)
        })
        
        #print(f"  {spec['name']:12s}: Woman Coef={coef:.4f} (SE={se_coef:.4f}), AME={ame:.4f} (SE={se_ame:.4f}), R²={model.rsquared:.3f}, N={int(model.nobs)}")

In [ ]:
# Generate LaTeX table with 6 panels
latex_decomp_intent = []
latex_decomp_intent.append("\\begin{table}[htbp]")
latex_decomp_intent.append("\\centering")
latex_decomp_intent.append("\\caption{Sequential Decomposition of Gender Gap by Intent Type}")
latex_decomp_intent.append("\\label{tab:decomp_by_intent}")
latex_decomp_intent.append("\\small")
latex_decomp_intent.append("\\begin{tabular}{l" + "c" * 6 + "}")
latex_decomp_intent.append("\\toprule")
latex_decomp_intent.append(" & Baseline & +Demo & +Edu & +Exp & +TechLit & +TechEdu \\\\")
latex_decomp_intent.append("\\midrule")

for intent_var in intent_types:
    intent_label = intent_labels[intent_var]
    results = decomp_results[intent_var]
    
    # Panel header
    latex_decomp_intent.append(f"\\multicolumn{{7}}{{l}}{{\\textit{{Panel: {intent_label}}}}} \\\\")
    
    # Woman coefficient row
    coef_row = "Woman"
    for r in results:
        if not np.isnan(r['coef']):
            stars = '***' if r['pval_coef'] < 0.01 else '**' if r['pval_coef'] < 0.05 else '*' if r['pval_coef'] < 0.1 else ''
            coef_str = f"{r['coef']:.3f}$^{{{stars}}}$" if stars else f"{r['coef']:.3f}"
        else:
            coef_str = "—"
        coef_row += f" & {coef_str}"
    latex_decomp_intent.append(coef_row + " \\\\")
    
    # SE row (in parentheses)
    se_row = ""
    for r in results:
        if not np.isnan(r['se_coef']):
            se_row += f" & ({r['se_coef']:.3f})"
        else:
            se_row += " & —"
    latex_decomp_intent.append(se_row + " \\\\")
    
    # AME row
    ame_row = "AME"
    for r in results:
        if not np.isnan(r['ame']):
            ame_row += f" & {r['ame']:.3f}"
        else:
            ame_row += " & —"
    latex_decomp_intent.append(ame_row + " \\\\")
    
    # AME SE row (in parentheses)
    se_ame_row = ""
    for r in results:
        if not np.isnan(r['se_ame']):
            se_ame_row += f" & ({r['se_ame']:.3f})"
        else:
            se_ame_row += " & —"
    latex_decomp_intent.append(se_ame_row + " \\\\")
    
    # R² row
    r2_row = "$R^2$"
    for r in results:
        r2_row += f" & {r['r2']:.3f}"
    latex_decomp_intent.append(r2_row + " \\\\")
    
    # N row
    n_row = "N"
    for r in results:
        n_row += f" & {r['n']:,}"
    latex_decomp_intent.append(n_row + " \\\\")
    
    # Separator between panels (except after last)
    if intent_var != intent_types[-1]:
        latex_decomp_intent.append("\\midrule")

latex_decomp_intent.append("\\bottomrule")
latex_decomp_intent.append("\\end{tabular}")
latex_decomp_intent.append("\\begin{tablenotes}")
latex_decomp_intent.append("\\small")
latex_decomp_intent.append("\\item \\textit{Notes:} Sequential decomposition showing Woman coefficient and AME.")
latex_decomp_intent.append("\\item Standard errors in parentheses. AME SE from bootstrap (1000 iterations).")
latex_decomp_intent.append("\\item Baseline: Gender only. +Demo: Age, Geography, Income. +Edu: Education level.")
latex_decomp_intent.append("\\item +Exp: LT experience. +TechLit: LT literacy. +TechEdu: STEM education.")
latex_decomp_intent.append("\\item $^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.1$")
latex_decomp_intent.append("\\end{tablenotes}")
latex_decomp_intent.append("\\end{table}")

latex_decomp_intent_str = "\\n".join(latex_decomp_intent)
# print("\n" + "=" * 80)
# print("LATEX TABLE - SEQUENTIAL DECOMPOSITION BY INTENT (6 PANELS)")
# print("=" * 80)
# print(latex_decomp_intent_str)

with open(results_dir / 'table_decomp_by_intent.tex', 'w') as f:
    f.write(latex_decomp_intent_str)
print("\n✓ LaTeX table saved to: results/table_decomp_by_intent.tex")

In [ ]:
# Save sequential decomposition results to TSV
decomp_by_intent_data = []
for intent_var in intent_types:
    for result in decomp_results[intent_var]:
        decomp_by_intent_data.append({
            'intent': intent_labels[intent_var],
            'intent_var': intent_var,
            'specification': result['spec'],
            'woman_coef': result['coef'],
            'woman_se': result['se_coef'],
            'woman_pval': result['pval_coef'],
            'ame': result['ame'],
            'ame_se': result['se_ame'],
            'r_squared': result['r2'],
            'n_obs': result['n']
        })

df_decomp_by_intent = pd.DataFrame(decomp_by_intent_data)
df_decomp_by_intent.to_csv(results_dir / 'decomposition_by_intent.tsv', sep='\t', index=False)
print("\n✓ Sequential decomposition data saved to: results/decomposition_by_intent.tsv")
print(f"  Shape: {df_decomp_by_intent.shape}")